# Сезонное преобразование спутниковых изображений

Парная регрессия **лето → зима** и **зима → лето**, U-Net / ResNet34.
В исходном эксперименте классы назывались DiffusionModel, но шумовой процесс и timestep
не участвовали в обучении. Поэтому здесь используется название seasonal translation.

Сохранённые изображения и loss старого запуска доступны в истории Git. Новый протокол
разделяет города до нарезки на тайлы и сравнивает прогноз с парным изображением другого сезона.
Новых результатов обучения на реальных данных пока нет. Подробности — README.md и ../AUDIT.md.

Установите requirements-torch.txt из корня репозитория. Данные: DATA_ROOT с парами
<city>_summer.jpg и <city>_winter.jpg. Совпадение размеров не доказывает географическую регистрацию.

In [ ]:
from pathlib import Path
import os
import importlib.util
import sys

folder = Path.cwd()
if not (folder / 'train.py').exists():
    folder = folder / 'Generation of satellite images'
spec = importlib.util.spec_from_file_location('seasonal_training', folder / 'train.py')
training = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = training
spec.loader.exec_module(training)
DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data/satellite'))

In [ ]:
# Отдельные модели для двух направлений. Данные и seed разбиения одинаковы.
# pretrained загрузит веса ImageNet; без него encoder обучается с нуля.
training.main(['--data-dir', str(DATA_ROOT), '--output-dir', 'results/summer-to-winter',
               '--direction', 'summer-to-winter', '--epochs', '50', '--pretrained'])

In [ ]:
training.main(['--data-dir', str(DATA_ROOT), '--output-dir', 'results/winter-to-summer',
               '--direction', 'winter-to-summer', '--epochs', '50', '--pretrained'])

In [ ]:
# Визуализация только отложенного города: вход / эталон / прогноз лучшей эпохи.
from PIL import Image
from IPython.display import display
for direction in ['summer-to-winter', 'winter-to-summer']:
    print(direction)
    display(Image.open(Path('results') / direction / 'validation_example.png'))

## Финальное тестирование

После фиксации гиперпараметров выполните финальный запуск в новой папке с `--evaluate-test`.
Города теста не участвуют в выборе эпохи. MAE/MSE относятся к нормированным пикселям [0, 1],
а не к вероятности реалистичности. Сравните их с identity baseline в metrics.json.
Не используйте изображение входного сезона как целевой эталон. Один город на split при
минимальных трёх городах даёт только техническую проверку, не устойчивую оценку обобщения.